> VIGOR 2.12.1 example. Install with `pip install "VIGOR[io,progress,plot] @ git+https://github.com/sanchestm/VIGOR"` (see the [README](../../../README.md)). Outputs are cleared: this notebook was checked against the 2.12.1 API but not re-executed for this release.

# LD-aware fine mapping of significant HS peaks

VIGOR 2.12.1. Run 06 first. This bounds a candidate panel to 256 SNPs; PIPs are conditional on that panel. The synthetic tutorial also demonstrates SHAP attribution.


In [ ]:
from pathlib import Path
import sys, os, json, warnings
import numpy as np
import pandas as pd
import xarray as xr

# Find ROOT/vigor_run/project_config.py regardless of notebook cwd.
def _find_run(start=None):
    p=Path(os.environ.get('VIGOR_PROJECT_ROOT', os.environ.get('DOMINO_PROJECT_ROOT', Path.cwd()))) if start is None else Path(start)
    for q in [p,*p.parents]:
        cand=q/'vigor_run'/'project_config.py'
        if cand.exists(): return cand.parent
        if q.name=='vigor_run' and (q/'project_config.py').exists(): return q
    raise FileNotFoundError('Set VIGOR_PROJECT_ROOT or place vigor_run under the project root.')
RUN=_find_run(); sys.path.insert(0,str(RUN))
from project_config import *
import vigor
print('Project root:',ROOT)
print('VIGOR:',vigor.__version__)
assert tuple(map(int, vigor.__version__.split('.')[:3])) >= (2, 12, 1), vigor.__version__


In [ ]:
gwas=vigor.open_gwas_zarr(GWAS_STORE)
loci=vigor.build_loci(gwas,metric="neglog_p",threshold=5.,window_bp=500_000,merge_traits=False)
display(loci.head())
if len(loci):
    locus=loci.iloc[0]
    geno=vigor.open_genotype_dataset(PLINK_PREFIX,source_format="plink1")
    # Coordinate mask is bounded to one locus; rank limits must be reported.
    chromosome=str(locus.get("chrom",locus.get("chromosome")))
    start=int(locus.get("start",locus.get("locus_start")))
    end=int(locus.get("end",locus.get("locus_end")))
    take=np.flatnonzero((np.asarray(geno.chromosome,dtype=object).astype(str)==chromosome)&(np.asarray(geno.position)>=start)&(np.asarray(geno.position)<=end))
    train=pd.read_csv(TRAIN_CSV,index_col=0);train.index=train.index.astype(str)
    take=take[:256]
    if len(take):
        values=geno.GT.sel(sample=train.index.astype(str)).isel(variant=take).compute()
        region=pd.DataFrame(values.values,index=train.index,columns=np.asarray(values.variant,dtype=object).astype(str))
        ld=vigor.compute_variant_ld(region.loc[train.notna().any(axis=1)],statistic="both")
        fm=vigor.fine_map_region(region,train.iloc[:,:1],methods=("ridge","bayes"),ld=ld)
        display(fm.ranking if hasattr(fm,"ranking") else fm)
else:print("No significant locus at the declared threshold; no artificial peak is created.")
